In [1]:
# You are a procurement analyst at a global manufacturer. 
# Your VP of Supply Chain has asked for a 12-month OTD trend report across 6 suppliers. 
# She wants to know which suppliers are improving, which are declining, and which are volatile. 
# She also wants to know if any supplier has missed the 85% OTD target for 3 or more consecutive months.
# No method hints. Draw on everything from the curriculum.

# import libraries
import pandas as pd
import numpy as np
import plotly.express as px

# set the seed
np.random.seed(7)

# dataframe

suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMfg", "EastCoast", "WestSupply"]
months = pd.date_range(start="2024-01-01", periods=12, freq="ME")

rows = []
for supplier in suppliers:
    base_otd = np.random.uniform(0.75, 0.95)
    trend = np.random.uniform(-0.02, 0.02)
    for i, month in enumerate(months):
        noise = np.random.uniform(-0.08, 0.08)
        otd = round(min(max(base_otd + trend * i + noise, 0.5), 1.0), 4)
        deliveries = np.random.randint(20, 80)
        on_time = round(otd * deliveries)
        rows.append({
            "supplier": supplier,
            "month": month,
            "deliveries": deliveries,
            "on_time": on_time,
            "otd_rate": round(on_time / deliveries * 100, 2)
        })

df = pd.DataFrame(rows)
print(df.shape)
print(df.head(12))


(72, 5)
   supplier      month  deliveries  on_time  otd_rate
0      Acme 2024-01-31          43       32     74.42
1      Acme 2024-02-29          77       59     76.62
2      Acme 2024-03-31          28       22     78.57
3      Acme 2024-04-30          62       45     72.58
4      Acme 2024-05-31          59       46     77.97
5      Acme 2024-06-30          68       56     82.35
6      Acme 2024-07-31          64       56     87.50
7      Acme 2024-08-31          75       62     82.67
8      Acme 2024-09-30          39       32     82.05
9      Acme 2024-10-31          25       22     88.00
10     Acme 2024-11-30          75       71     94.67
11     Acme 2024-12-31          79       64     81.01


In [2]:
# Part 1 -- Validate the Dataset
# Review the shape, null values, duplicates, dtypes, and business logic for the OTD data
# df.shape[1] accesses the second element --> the columns --> 9
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

# check the data for null values
# begin by defining nulls
nulls = df.isnull().sum()

# write an f-string
print(f"\nNull values per column:\n{nulls}")
print(f"Total nulls: {nulls.sum()}") 

# check for duplicate values
dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")

# review the dataframe types
print(f"\nColumn dtypes:\n{df.dtypes}")

# descriptive stats of the shape of the dataframe
print(f"\nDescriptive Statistics:\n {df.describe().round(2)}")

# business logic -- sanity check on on_time with deliveries
logicone = df[df["on_time"] > df["deliveries"]]
print(f"\nBusiness Logic Violations (on_time > deliveries): {len(logicone)}")

# business logic -- sanity check on otd_rate
logictwo = df[df["otd_rate"] <= 50]
print(f"\nBusiness Logic Violations (otd_rate <= 50): {len(logictwo)}")


Dataset shape: 72 rows x 5 columns

Null values per column:
supplier      0
month         0
deliveries    0
on_time       0
otd_rate      0
dtype: int64
Total nulls: 0

Duplicate rows: 0

Column dtypes:
supplier                 str
month         datetime64[us]
deliveries             int64
on_time                int64
otd_rate             float64
dtype: object

Descriptive Statistics:
                      month  deliveries  on_time  otd_rate
count                   72       72.00    72.00     72.00
mean   2024-07-15 20:00:00       51.83    44.60     85.76
min    2024-01-31 00:00:00       23.00    18.00     67.21
25%    2024-04-22 12:00:00       34.50    28.50     78.64
50%    2024-07-15 12:00:00       54.00    45.00     86.14
75%    2024-10-07 18:00:00       68.00    57.50     91.10
max    2024-12-31 00:00:00       79.00    74.00    100.00
std                    NaN       18.43    16.77      8.17

Business Logic Violations (on_time > deliveries): 0

Business Logic Violations (otd_rate 

In [3]:
# Part 2 — Monthly OTD summary:
# Using .pivot_table(), build a supplier × month matrix showing otd_rate. 
# # Months as columns, 
# suppliers as rows. 
# Round to 1 decimal.

# build the pivot-table
# .dt.strftime("%b") converts a datetime to a short month name — "Jan", "Feb"
# once you convert months to string you will need to order them

# programmatic
month_order = pd.date_range("2024-01-01", periods=12, freq="ME").strftime("%b-%Y").tolist()

# hard code
# month_order = [
#    "Jan-2024", "Feb-2024", "Mar-2024", "Apr-2024",
#    "May-2024", "Jun-2024", "Jul-2024", "Aug-2024",
#    "Sep-2024", "Oct-2024", "Nov-2024", "Dec-2024"
#]

month_summary = (
    df
    .assign(
        month_label = lambda x: pd.Categorical(
        x["month"].dt.strftime("%b-%Y"),
        categories=month_order,
        ordered=True
    )
    )
    .pivot_table(
        index="supplier",
        columns="month_label",
        values="otd_rate",
        aggfunc="mean"
    )
    .round(1)
)

# print the pivot_table
print(month_summary)

month_label  Jan-2024  Feb-2024  Mar-2024  Apr-2024  May-2024  Jun-2024  \
supplier                                                                  
Acme             74.4      76.6      78.6      72.6      78.0      82.4   
EastCoast        84.6      86.1      88.9      93.1      88.0      85.1   
FastParts        87.2      84.0      86.2      72.6      77.1      78.7   
GlobalCo         74.1      89.7      82.6      88.2      80.6      81.6   
PrimeMfg         78.3      85.7      78.8      78.3      84.4      78.6   
WestSupply       88.7      90.9      87.7      87.8      98.3      93.7   

month_label  Jul-2024  Aug-2024  Sep-2024  Oct-2024  Nov-2024  Dec-2024  
supplier                                                                 
Acme             87.5      82.7      82.0      88.0      94.7      81.0  
EastCoast        91.7      95.5      95.4      93.6     100.0     100.0  
FastParts        76.9      75.0      67.2      78.3      75.0      74.3  
GlobalCo         88.9      86

In [4]:
# Part 3 — Trend classification:
# For each supplier calculate:

# avg_otd — average OTD rate across all 12 months
# first_half_avg — average OTD for months 1–6 (Jan–Jun)
# second_half_avg — average OTD for months 7–12 (Jul–Dec)
# trend — "improving" if second half > first half by more than 2pp, "declining" if first half > second half by more than 2pp, "stable" otherwise

# Use np.select() for the trend classification.

# define time-periods
# programmatic "yyyy-mm-dd"
# jan_dec = pd.date_range("2024-01-01", periods=12, freq="ME").strftime("%b-%Y").tolist()
# jan_jun = pd.date_range("2024-01-01", periods=6, freq="ME").strftime("%b-%Y").tolist()
# jul_dec = pd.date_range("2024-07-01", periods=6, freq="ME").strftime("%b-%Y").tolist()
# print(jan_dec)
# print(jan_jun)
# print(jul_dec)

# add a half-year label for months 1-6 with "first" and second-half year label for months 7-12 with "second"
df = (
    df
    .assign(
        half = lambda x: x["month"].dt.month.map(lambda m: "first" if m <=6 else "second")
        )
    )

# calculate the averages
half_avg = (
    df
    .pivot_table(
        index="supplier",
        columns="half",
        values="otd_rate",
        aggfunc="mean"
    )
    .round(2)
    .rename(columns={"first": "first_half_2024_avg", "second": "second_half_2024_avg"})
)

print(half_avg)

overall_avg = (
    df
    .pivot_table(
        index="supplier",
        values="otd_rate",
        aggfunc="mean"
    )
    .round(2)
    .rename(columns={"otd_rate":"overall_2024_avg"})
)
print(overall_avg)

# join both pivot-table outputs into a single dataframe
scorecard = (
    half_avg
    .join(
        other=overall_avg,
        how="left"
    )
)

print(scorecard)

# classify the trend
# define your condiitons and choices for np.select
# np.select evaluates each condition in order
# you are comparing percentage points -- if there is an increase of > 2 pp from 1st half to 2nd half then improving
conditions = [
    scorecard["second_half_2024_avg"] - scorecard["first_half_2024_avg"] > 2,
    scorecard["first_half_2024_avg"] - scorecard["second_half_2024_avg"] > 2,
]

choices = ["improving",
           "declining"
           ]

scorecard = (
    scorecard
    .assign(
        trend = np.select(condlist=conditions,
                          choicelist=choices,
                          default="stable")
    )
)
print(scorecard)

half        first_half_2024_avg  second_half_2024_avg
supplier                                             
Acme                      77.08                 85.98
EastCoast                 87.64                 96.03
FastParts                 80.96                 74.45
GlobalCo                  82.81                 91.06
PrimeMfg                  80.66                 82.37
WestSupply                91.18                 98.96
            overall_2024_avg
supplier                    
Acme                   81.53
EastCoast              91.83
FastParts              77.70
GlobalCo               86.93
PrimeMfg               81.52
WestSupply             95.07
            first_half_2024_avg  second_half_2024_avg  overall_2024_avg
supplier                                                               
Acme                      77.08                 85.98             81.53
EastCoast                 87.64                 96.03             91.83
FastParts                 80.96                 

In [5]:
# Part 4
# Identify Consecutive Misses: 
# Flag suppliers that miss the 85% OTD Target for 3 or more consecutive months
# the goal is to detect consecutive streaks

# use .shift()
# df.shift(1, axis=0)   # shift rows down — default
# df.shift(1, axis=1)   # shift columns right

# sort the datafram by supplier then month --> this needs to reset by supplier
df = (
    df
    .sort_values(by=["supplier", "month"])
    .reset_index(drop=True) # reset_index resets the integer index after sorting
    )

# flag months that are misses
df = (
    df
    .assign(
        miss = lambda x: x["otd_rate"] < 85,
        miss_lagone = lambda x: x.groupby(by="supplier")["miss"].shift(1), # shifts down one row --> each row is a unique month
        miss_lagtwo = lambda x: x.groupby(by="supplier")["miss"].shift(2),
        cons_miss = lambda x: x["miss"] & x["miss_lagone"] & x["miss_lagtwo"]
    )
)

# identify suppliers with at leat one consecutive miss
suppliers_one = (
    df[df["cons_miss"] == True]["supplier"]
    .unique()
    .tolist()
)

# f-string
print(f"Suppliers with 3+ consecutive misses: {suppliers_one}")
print(df[df["cons_miss"] == True][["supplier", "month", "otd_rate", "cons_miss"]])

Suppliers with 3+ consecutive misses: ['Acme', 'FastParts', 'PrimeMfg']
     supplier      month  otd_rate  cons_miss
2        Acme 2024-03-31     78.57       True
3        Acme 2024-04-30     72.58       True
4        Acme 2024-05-31     77.97       True
5        Acme 2024-06-30     82.35       True
29  FastParts 2024-06-30     78.67       True
30  FastParts 2024-07-31     76.92       True
31  FastParts 2024-08-31     75.00       True
32  FastParts 2024-09-30     67.21       True
33  FastParts 2024-10-31     78.26       True
34  FastParts 2024-11-30     75.00       True
35  FastParts 2024-12-31     74.29       True
52   PrimeMfg 2024-05-31     84.38       True
53   PrimeMfg 2024-06-30     78.57       True
54   PrimeMfg 2024-07-31     75.34       True
55   PrimeMfg 2024-08-31     81.25       True


In [11]:
# Part 5
# Rolling 3-month std of otd_rate per supplier using groupby + rolling
# Flag months where rolling std exceeds 5 percentage points as "high_volatility"

# sort the dataframe
df = (
    df
    .assign(
        rolling_std = lambda x: (x.groupby(by="supplier")["otd_rate"]
                                 .transform(lambda s: s.rolling(3).std().round(2))
                                 ),
        high_volatility = lambda x: x["rolling_std"] > 5
    )
)
print(df)

# f-string for answers
print(f"\nHigh volatility months: {df['high_volatility'].sum()}")
print(df[df["high_volatility"] == True][["supplier", "month", "otd_rate", "rolling_std"]])



      supplier      month  deliveries  on_time  otd_rate    half   miss  \
0         Acme 2024-01-31          43       32     74.42   first   True   
1         Acme 2024-02-29          77       59     76.62   first   True   
2         Acme 2024-03-31          28       22     78.57   first   True   
3         Acme 2024-04-30          62       45     72.58   first   True   
4         Acme 2024-05-31          59       46     77.97   first   True   
..         ...        ...         ...      ...       ...     ...    ...   
67  WestSupply 2024-08-31          55       55    100.00  second  False   
68  WestSupply 2024-09-30          73       73    100.00  second  False   
69  WestSupply 2024-10-31          53       53    100.00  second  False   
70  WestSupply 2024-11-30          72       72    100.00  second  False   
71  WestSupply 2024-12-31          65       65    100.00  second  False   

   miss_lagone miss_lagtwo  cons_miss  rolling_std  high_volatility  
0          NaN         NaN   

In [ ]:
# Part 6
# Build a multi-line chart showing each supplier's OTD rate over 12 months 
# with the 85% target line added as a horizontal reference line.

fig = px.line(
    df,
    x="month",
    y="otd_rate",
    color="supplier",
    title="Supplier OTD Rate L12 Months vs. 85% Target Rate"
)

# add the 85% target line
fig.add_hline(
    y=85,
    line_dash="dash",
    line_color="red",
    annotation_text="85% Target",
    annotation_position="top left"
)

fig.show()
